In [1]:
import gymnasium as gym
import numpy as np
import random
import cv2
from collections import deque
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
# ---------- 1. 预处理包装器: 灰度 + Resize 到 (1, 96, 96) ----------
class GrayscaleResizeWrapper(gym.ObservationWrapper):
    def __init__(self, env: gym.Env, width: int = 96, height: int = 96):
        super().__init__(env)
        self.width, self.height = width, height
        self.observation_space = gym.spaces.Box(
            low=0,
            high=255,
            shape=(1, self.height, self.width),
            dtype=np.uint8,
        )

    def observation(self, obs):
        gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        resized = cv2.resize(gray, (self.width, self.height), interpolation=cv2.INTER_AREA)
        return np.expand_dims(resized, axis=0)


In [3]:
# ---------- 2. 离散动作映射表 (5 actions) ----------
ACTION_TABLE = np.array(
    [
        [0.0, 0.0, 0.0],  # 0: idle
        [-1.0, 0.0, 0.0],  # 1: steer left
        [1.0, 0.0, 0.0],  # 2: steer right
        [0.0, 1.0, 0.0],  # 3: gas
        [0.0, 0.0, 0.8],  # 4: brake
    ],
    dtype=np.float32,
)


def map_action(idx: int) -> np.ndarray:
    return ACTION_TABLE[idx]


In [4]:
# ---------- 3. DQN 网络 ----------
class DQN(nn.Module):
    def __init__(self, n_actions: int = 5):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 8, 4),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, 1),
            nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 9 * 9, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions),
        )

    def forward(self, x):
        x = x / 255.0  # 归一化到 0‑1
        x = self.conv(x)
        return self.fc(x)

In [5]:
# ---------- 4. 环境构造 ----------

def make_env(seed: int = 0):
    env = gym.make(
        "CarRacing-v3",
        render_mode="rgb_array",  # 训练时用数组; 评估可改 "human"
        lap_complete_percent=0.95,
        domain_randomize=False,
        continuous=False,
    )
    env.reset(seed=seed)
    env = GrayscaleResizeWrapper(env)
    return env

In [6]:
# ---------- 5. 训练超参数 ----------
MAX_EPISODES = 500
MAX_STEPS = 2_000  # 每局最多步数
BUFFER_SIZE = 20_000
BATCH_SIZE = 64
GAMMA = 0.99
LEARNING_RATE = 1e-4
TARGET_UPDATE = 1_000  # 每多少步同步目标网络
EPS_START, EPS_END, EPS_DECAY = 1.0, 0.1, 0.995
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)


In [7]:
# ---------- 6. 经验回放 ----------
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, *transition):
        self.buffer.append(tuple(transition))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = map(np.stack, zip(*batch))
        return (
            torch.tensor(states, dtype=torch.float32, device=device),
            torch.tensor(actions, dtype=torch.long, device=device),
            torch.tensor(rewards, dtype=torch.float32, device=device),
            torch.tensor(next_states, dtype=torch.float32, device=device),
            torch.tensor(dones, dtype=torch.bool, device=device),
        )

    def __len__(self):
        return len(self.buffer)


In [8]:
# ---------- 7. 主训练循环 ----------

def preprocess(obs: np.ndarray) -> torch.Tensor:
    """
    将 (1,96,96) numpy -> torch.float32 (1, 1, 96, 96) 并放到 device
    """
    return torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)

def train():
    env = make_env(seed=42)           # CarRacing‑v3, continuous=False
    n_actions = 5

    q_net     = DQN(n_actions).to(device)
    target_net = DQN(n_actions).to(device)
    target_net.load_state_dict(q_net.state_dict())
    optimizer = optim.Adam(q_net.parameters(), lr=LEARNING_RATE)

    buffer = ReplayBuffer(BUFFER_SIZE)

    epsilon      = EPS_START
    global_step  = 0

    for episode in range(1, MAX_EPISODES + 1):
        obs, _ = env.reset()
        state  = preprocess(obs)                  # shape: (1,1,96,96)
        episode_reward = 0.0

        for _ in range(MAX_STEPS):
            # ε‑greedy
            if random.random() < epsilon:
                action_idx = env.action_space.sample()   # 0‑4
            else:
                with torch.no_grad():
                    action_idx = q_net(state).argmax(1).item()

            # === 与环境交互 ===
            next_obs, reward, terminated, truncated, _ = env.step(action_idx)
            done = terminated or truncated
            next_state = preprocess(next_obs)

            # 存入 replay buffer (存 numpy，维度 (1,96,96))
            buffer.push(obs, action_idx, reward, next_obs, done)

            # 更新指针
            obs   = next_obs
            state = next_state
            episode_reward += reward
            global_step += 1

            # ---------- 参数更新 ----------
            if len(buffer) >= BATCH_SIZE:
                states, actions, rewards, next_states, dones = buffer.sample(BATCH_SIZE)
                # shapes: states -> (B,1,96,96)
                q_values = q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

                with torch.no_grad():
                    max_next_q = target_net(next_states).max(1)[0]
                    targets = rewards + GAMMA * max_next_q * (~dones)

                loss = nn.functional.smooth_l1_loss(q_values, targets)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # ---------- 同步目标网络 ----------
            if global_step % TARGET_UPDATE == 0:
                target_net.load_state_dict(q_net.state_dict())

            if done:
                break

        # ε 衰减
        epsilon = max(EPS_END, epsilon * EPS_DECAY)
        print(f"Ep {episode:03d} | reward {episode_reward:7.2f} | epsilon {epsilon:.3f}")

        # 周期性保存模型
        if episode % 50 == 0:
            torch.save(q_net.state_dict(), MODEL_DIR / f"dqn_carracing_ep{episode}.pth")

    env.close()


In [9]:
train()

RuntimeError: mat1 and mat2 shapes cannot be multiplied (64x4096 and 5184x512)